# Custom Dataset Tokenization

This notebook is the easy path for turning a text dataset into the format used by this repo.

It does four things:

1. Selects a data source: Hugging Face, local JSONL, local text files, or a folder of text files.
2. Trains or reuses a byte-BPE tokenizer.
3. Builds packed `token_shards` for causal-LM training.
4. Writes reusable YAML configs so the same run can be launched from the terminal.

Start with `MODE = "smoke"`. Once that works, switch to `MODE = "full"`.

## 1. Setup

Run this cell first. It finds the repo root and imports the same helper code used by `scripts/data/build_token_shards.py`.

In [ ]:
from pathlib import Path
from itertools import islice
import json
import os
import shutil
import sys

import yaml

sys.dont_write_bytecode = True

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "scripts" / "build_token_shards.py").exists():
            return candidate
    raise RuntimeError("Could not find the repo root. Start Jupyter inside babylm-baby-steps.")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"
SRC = ROOT / "src"
for path in [SCRIPTS, SRC]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from babysteps.cli.build_token_shards import (
    build_token_shards,
    iter_dataset_rows,
    local_dataset_files,
    train_or_load_tokenizer,
)
from babysteps.data.token_shards import build_token_shard_loader

print(f"Repo root: {ROOT}")

## 2. Choose Your Dataset

Edit this one cell. The default creates a tiny local demo corpus so the notebook can run end to end before you point it at real data.

In [ ]:
# Keep this True for a first run. Set it to False when using your real data.
USE_DEMO_DATA = True

# Choose one: "hf", "local_jsonl", "local_text_files", "local_text_dir".
DATA_SOURCE = "local_text_dir"

# Use "smoke" first. Use "full" after the smoke build looks right.
MODE = "smoke"

# A short lowercase name used in output folders and config files.
PROJECT_NAME = "demo_custom_dataset"

# Hugging Face dataset settings, used when DATA_SOURCE = "hf".
HF_DATASET = "BabyLM-community/BabyLM-2026-Strict"
HF_CONFIG = "default"
HF_SPLIT = "train"

# Local JSONL settings, used when DATA_SOURCE = "local_jsonl".
# Each line should be a JSON object with a text field, for example: {"text": "hello"}
LOCAL_JSONL = Path("/path/to/train.jsonl")

# Local text-file settings.
LOCAL_TEXT_FILES = [Path("/path/to/file1.txt"), Path("/path/to/file2.txt")]
LOCAL_TEXT_DIR = ROOT / "cache" / "notebook_demo_text"
TEXT_GLOB = "*.txt"
RECURSIVE = False

# For local_text: "file" means each file is one document; "line" means each non-empty line is one document.
DOCUMENT_MODE = "file"

# Field and tokenizer/shard settings.
TEXT_FIELD = "text"
VOCAB_SIZE = 16_384
MIN_FREQUENCY = 2
MODEL_MAX_LENGTH = 1024
SHARD_TOKENS = 1_000_000
VAL_SHARDS = 1
BATCH_DOCS = 512

# Smoke mode caps. Smaller shards make it easy to verify a train shard exists.
SMOKE_MAX_DOCS = 1_000
SMOKE_MAX_SHARDS = 4
SMOKE_SHARD_TOKENS = 4_096

# Rerun-friendly defaults. Set OVERWRITE_TOKENIZER=True if you change the corpus or vocab settings.
OVERWRITE_TOKENIZER = False
OVERWRITE_SHARDS = True

default_cache = ROOT / "cache"
OUTPUT_BASE = Path(os.environ.get("BABYSTEPS_CACHE_DIR", str(default_cache))) / "custom_datasets"

## 3. Optional Demo Data

This creates a small folder of `.txt` files when `USE_DEMO_DATA = True`. Skip it by setting `USE_DEMO_DATA = False` above.

In [ ]:
if USE_DEMO_DATA:
    PROJECT_NAME = "demo_custom_dataset"
    DATA_SOURCE = "local_text_dir"
    MODE = "smoke"
    DOCUMENT_MODE = "file"
    TEXT_GLOB = "*.txt"
    LOCAL_TEXT_DIR.mkdir(parents=True, exist_ok=True)
    demo_docs = {
        "stories_1.txt": "A small model reads simple stories. The stories repeat useful patterns.\n" * 500,
        "stories_2.txt": "Children ask questions. Caregivers answer with patient explanations.\n" * 500,
        "stories_3.txt": "The dataset builder packs tokens into shards for fast training.\n" * 500,
    }
    for filename, text in demo_docs.items():
        (LOCAL_TEXT_DIR / filename).write_text(text, encoding="utf-8")
    print(f"Demo text files written under: {LOCAL_TEXT_DIR}")
else:
    print("Using your configured data source.")

## 4. Build The Config

This converts your choices into the same YAML-shaped dictionary used by the CLI scripts.

In [ ]:
def require_real_path(path, label):
    path = Path(path).expanduser()
    if str(path).startswith("/path/to"):
        raise ValueError(f"Edit {label}; it still points at {path!s}")
    return path

def source_config():
    if DATA_SOURCE == "hf":
        cfg = {
            "kind": "hf",
            "name": HF_DATASET,
            "split": HF_SPLIT,
            "text_field": TEXT_FIELD,
            "streaming": True,
        }
        if HF_CONFIG:
            cfg["config"] = HF_CONFIG
        return cfg

    if DATA_SOURCE == "local_jsonl":
        return {
            "kind": "local_jsonl",
            "files": {"train": str(require_real_path(LOCAL_JSONL, "LOCAL_JSONL"))},
            "split": "train",
            "text_field": TEXT_FIELD,
            "encoding": "utf-8",
        }

    if DATA_SOURCE == "local_text_files":
        return {
            "kind": "local_text",
            "files": [str(require_real_path(path, "LOCAL_TEXT_FILES")) for path in LOCAL_TEXT_FILES],
            "text_field": TEXT_FIELD,
            "document_mode": DOCUMENT_MODE,
            "encoding": "utf-8",
        }

    if DATA_SOURCE == "local_text_dir":
        return {
            "kind": "local_text",
            "dir": str(require_real_path(LOCAL_TEXT_DIR, "LOCAL_TEXT_DIR")),
            "glob": TEXT_GLOB,
            "recursive": bool(RECURSIVE),
            "text_field": TEXT_FIELD,
            "document_mode": DOCUMENT_MODE,
            "encoding": "utf-8",
        }

    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE!r}")

if MODE not in {"smoke", "full"}:
    raise ValueError("MODE must be 'smoke' or 'full'.")

run_name = f"{PROJECT_NAME}_{MODE}_byte_bpe_{VOCAB_SIZE}"
tokenizer_dir = OUTPUT_BASE / "tokenizers" / run_name
shards_dir = OUTPUT_BASE / "token_shards" / f"{run_name}_bos_eos"
effective_shard_tokens = SMOKE_SHARD_TOKENS if MODE == "smoke" else SHARD_TOKENS

config = {
    "name": f"{run_name}_shards",
    "dataset": source_config(),
    "tokenizer": {
        "kind": "byte_bpe",
        "vocab_size": int(VOCAB_SIZE),
        "min_frequency": int(MIN_FREQUENCY),
        "special_tokens": ["<unk>", "<s>", "</s>", "<pad>"],
        "unk_token": "<unk>",
        "bos_token": "<s>",
        "eos_token": "</s>",
        "pad_token": "<pad>",
        "model_max_length": int(MODEL_MAX_LENGTH),
        "train_batch_docs": 1000,
        "output_dir": str(tokenizer_dir),
    },
    "shards": {
        "output_dir": str(shards_dir),
        "shard_tokens": int(effective_shard_tokens),
        "val_shards": int(VAL_SHARDS),
        "batch_docs": int(BATCH_DOCS),
        "max_shards": None,
    },
}

print(yaml.safe_dump(config, sort_keys=False))

## 5. Preview The Data

This checks that files resolve and that each row/document has a `text` field.

In [ ]:
if config["dataset"]["kind"].startswith("local"):
    files = local_dataset_files(config["dataset"])
    print(f"Resolved {len(files)} local file(s).")
    for path in files[:10]:
        print(" -", path)
    if len(files) > 10:
        print(f" ... {len(files) - 10} more")

print("\nPreview documents:")
for idx, row in enumerate(islice(iter_dataset_rows(config["dataset"]), 3), start=1):
    text = str(row.get(TEXT_FIELD, ""))
    preview = text[:400].replace("\n", " ")
    print(f"[{idx}] {len(text):,} chars: {preview}")

## 6. Build The Tokenizer And Shards

This is the main step. In smoke mode it caps the document count and shard count so it finishes quickly.

In [ ]:
max_docs = SMOKE_MAX_DOCS if MODE == "smoke" else None
max_shards = SMOKE_MAX_SHARDS if MODE == "smoke" else None

tokenizer = train_or_load_tokenizer(
    config,
    overwrite=bool(OVERWRITE_TOKENIZER),
    max_docs=max_docs,
)

meta = build_token_shards(
    config,
    tokenizer,
    overwrite=bool(OVERWRITE_SHARDS),
    max_docs=max_docs,
    max_shards=max_shards,
)

print(json.dumps(meta, indent=2))

## 7. Quick Loader Check

This opens one train block if a train shard exists. If all tokens fit in the validation shard, reduce `VAL_SHARDS` or `SMOKE_SHARD_TOKENS`, or use more data.

In [ ]:
train_shards = sorted(shards_dir.glob("train_*.bin"))
val_shards = sorted(shards_dir.glob("val_*.bin"))
print(f"Tokenizer dir: {tokenizer_dir}")
print(f"Shards dir:    {shards_dir}")
print(f"Val shards:    {len(val_shards)}")
print(f"Train shards:  {len(train_shards)}")

if train_shards:
    loader, dataset = build_token_shard_loader(
        shards_dir,
        split="train",
        batch_size=1,
        seq_len=min(128, int(MODEL_MAX_LENGTH)),
        shuffle=False,
        max_blocks=1,
    )
    input_ids, labels, loss_mask, attention_mask = next(iter(loader))
    print("One train batch:")
    print(" input_ids", tuple(input_ids.shape))
    print(" labels   ", tuple(labels.shape))
    print(" decoded  ", tokenizer.decode(input_ids[0, :80].tolist()))
else:
    print("No train shard found yet. This can happen with very tiny smoke data.")

## 8. Save Reusable Config Files

This writes a tokenization config and a training config. The training config points at the tokenizer and shards created above.

In [ ]:
SAVE_CONFIGS = True

tokenization_config_path = ROOT / "configs" / "tokenization" / f"{run_name}_shards.yaml"
training_config_path = ROOT / "configs" / "train" / f"{PROJECT_NAME}_{MODE}_llama_token_shards.yaml"

if SAVE_CONFIGS:
    tokenization_config_path.parent.mkdir(parents=True, exist_ok=True)
    tokenization_config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

    template_path = ROOT / "configs" / "train" / "template_llama_token_shards.yaml"
    train_config = yaml.safe_load(template_path.read_text(encoding="utf-8"))
    train_config["run"]["name"] = f"{PROJECT_NAME}_{MODE}_llama_8l512_bpe{len(tokenizer)}_muon"
    train_config["data"]["data_dir"] = str(shards_dir)
    train_config["tokenizer"]["name"] = str(tokenizer_dir)
    train_config["model"]["vocab_size"] = int(len(tokenizer))
    train_config["model"]["context_length"] = int(MODEL_MAX_LENGTH)
    training_config_path.write_text(yaml.safe_dump(train_config, sort_keys=False), encoding="utf-8")

    print(f"Tokenization config: {tokenization_config_path}")
    print(f"Training config:     {training_config_path}")
else:
    print("SAVE_CONFIGS is False; nothing was written.")

## 9. Train From The Terminal

The notebook prepares the data. Training is usually better from a terminal or job launcher.

In [ ]:
relative_train_config = training_config_path.relative_to(ROOT)
print("Run this from the repo root:")
print(f"python scripts/train/train_causal_lm.py --config {relative_train_config}")

if MODE == "smoke":
    print("\nAfter the smoke build looks good, set MODE = 'full' near the top and rerun the notebook.")

## Notes

- `token_shards` are stored as `uint16`, so tokenizer vocab size must be below 65,536.
- If you change the dataset or tokenizer settings, set `OVERWRITE_TOKENIZER = True` for the next run.
- If rerunning only the shard build, `OVERWRITE_SHARDS = True` is usually fine.
- For local text folders, `document_mode = "file"` treats each file as one document. `document_mode = "line"` treats each non-empty line as one document.
- For nested folders, set `RECURSIVE = True`.